In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path
from src.data.functions import json_to_dict

In [2]:
top_dir = Path().absolute()
PARENT_FILE_PATH = top_dir / "models" / "2025-07-22 B7H4 SAM 1000L Model"
model_config = json_to_dict(Path(PARENT_FILE_PATH, "B7H4_model_parameters.json"))

a_matrix = np.array(model_config["a_matrix"])
b_matrix = np.array(model_config["b_matrix"])

In [3]:
import numpy as np
from scipy.linalg import solve_lyapunov, svdvals
from scipy.signal import StateSpace, lsim

class InputSensitivityAnalyzer:
    def __init__(self, A, B, C, D, input_names=None, output_names=None):
        self.A, self.B, self.C, self.D = A, B, C, D
        self.n_states = A.shape[0]
        self.n_inputs = B.shape[1]
        self.n_outputs = C.shape[0]
        self.input_names = input_names or [f"u{i}" for i in range(self.n_inputs)]
        self.output_names = output_names or [f"y{i}" for i in range(self.n_outputs)]

    def dc_gain(self):
        """Steady-state gain matrix K: output change per unit input change."""
        return -self.C @ np.linalg.solve(self.A, self.B) + self.D

    def controllability_gramians_per_input(self):
        """Solve Lyapunov equation per input channel."""
        gramians = []
        for j in range(self.n_inputs):
            bj = self.B[:, j:j+1]
            # AW + WA^T + bb^T = 0  =>  W = solve_lyapunov(A, -bb^T)
            Wj = solve_lyapunov(self.A, -bj @ bj.T)
            gramians.append(Wj)
        return gramians

    def input_importance(self):
        """Rank inputs by multiple criteria."""
        K = self.dc_gain()
        gramians = self.controllability_gramians_per_input()

        results = []
        for j in range(self.n_inputs):
            dc_norm = np.linalg.norm(K[:, j])
            gram_trace = np.trace(gramians[j])
            gram_max_eig = np.max(np.real(np.linalg.eigvals(gramians[j])))
            results.append({
                'input': self.input_names[j],
                'dc_gain_norm': dc_norm,
                'gramian_trace': gram_trace,
                'gramian_max_eigenvalue': gram_max_eig,
            })

        # Sort by gramian trace (total controllability energy)
        return sorted(results, key=lambda x: x['gramian_trace'], reverse=True)

    def frequency_response_per_input(self, freqs):
        """||G_j(jw)|| across frequency range for each input."""
        I = np.eye(self.n_states)
        results = {name: [] for name in self.input_names}

        for omega in freqs:
            resolvent = np.linalg.solve(1j * omega * I - self.A, self.B)
            G_jw = self.C @ resolvent + self.D
            for j in range(self.n_inputs):
                results[self.input_names[j]].append(np.linalg.norm(G_jw[:, j]))

        return freqs, results

In [4]:
c_matrix = np.identity(a_matrix.shape[0])
d_matrix = np.zeros([a_matrix.shape[0], b_matrix.shape[1]])

sensitivity = InputSensitivityAnalyzer(a_matrix, b_matrix, c_matrix, d_matrix)
importance = sensitivity.input_importance()

importance

[{'input': 'u0',
  'dc_gain_norm': np.float64(1.5242465978549962),
  'gramian_trace': np.float64(0.6216101832238099),
  'gramian_max_eigenvalue': np.float64(0.3434513720071517)},
 {'input': 'u1',
  'dc_gain_norm': np.float64(1.057519878734524),
  'gramian_trace': np.float64(0.04455943721418107),
  'gramian_max_eigenvalue': np.float64(0.0255850990322685)},
 {'input': 'u2',
  'dc_gain_norm': np.float64(0.44930619296124447),
  'gramian_trace': np.float64(0.04294628425117082),
  'gramian_max_eigenvalue': np.float64(0.031097054165890935)}]

In [5]:
a_matrix

array([[ 3.69759277e-01,  3.30944279e-01,  2.28144847e-01,
        -7.68280677e-01,  1.09805248e-04],
       [-7.65528656e-01, -1.30849716e-01, -1.96431589e-01,
         5.88776091e-01,  4.37645976e-01],
       [-5.49118545e-01, -1.70177467e-01, -1.95359923e-01,
         5.04288125e-01,  2.57080919e-01],
       [ 1.64675476e+00,  6.45768442e-01,  1.00265602e+00,
        -3.41765229e+00,  1.86953013e-01],
       [ 2.67228678e-01, -1.52807991e-01,  5.75952451e-02,
         3.80431327e-01, -5.32332917e-01]])

In [6]:
b_matrix

array([[-0.04143855, -0.06175966, -0.01399058],
       [ 0.10573728,  0.06272336,  0.0807145 ],
       [ 0.07030324,  0.02332423,  0.06562333],
       [ 0.06394341, -0.16348696,  0.04842953],
       [ 0.11433907, -0.00314985, -0.14728266]])

: 